# 01 baseline

Step 1-3 - infrastructure, the frozen split, and plain steering.

Gate: B0/B1 must reproduce the old repo's baseline before anything is trained.

In [6]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import torch

from steering import io

print(f"python  {sys.version.split()[0]}")
print(f"torch   {torch.__version__}")
print(f"mps     {torch.backends.mps.is_available()}")
print(f"repo    {io.REPO_ROOT}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
python  3.12.0
torch   2.13.0
mps     True
repo    /Users/bg/Coding/steering-mvp


## Проверка модуля для управления сохраненными результатами (scr/io.py)

In [7]:
# First call computes; second loads. Then a changed config must refuse to reuse the stale file.
cfg = {"model": "gpt2", "layer": 6, "seed": 0, "version": 1}

io.run_or_load("_smoke", cfg, lambda: {"hello": "world"})
io.run_or_load("_smoke", cfg, lambda: {"hello": "world"})

try:
    io.run_or_load("_smoke", {**cfg, "seed": 1}, lambda: {"hello": "other"})
except io.CacheMismatch as e:
    print(f"\nrefused, as it should:\n{e}")

computed _smoke.json  (7d44b47fe7adc931)
cached  _smoke.json  (7d44b47fe7adc931)

refused, as it should:
_smoke.json was produced by a different config.
  seed: 0 -> 1

Pass force=True to recompute and overwrite, or rename the run.


In [8]:
for f in io.RESULTS.glob("_smoke.*"):
    f.unlink()

## Загрузка модели

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from steering import hooks

MODEL, LAYER = "gpt2", 6

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL).eval()

print(f"{MODEL}: {hooks.n_layers(model)} blocks, hook at layer {LAYER} "
      f"(= hidden_states[{hooks.hidden_state_index(LAYER)}])")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

gpt2: 12 blocks, hook at layer 6 (= hidden_states[7])


In [10]:
batch = tokenizer(["The capital of France is"], return_tensors="pt")

with hooks.ResidualHook(model, layer=LAYER, capture=True) as hook, torch.no_grad():
    model(**batch)

h = hook.captured[0][0]                     # [T, D]
print("norm per position:", [f"{v:.0f}" for v in h.norm(dim=-1)])
print("top outlier dims: ", h.abs().max(0).values.topk(3).indices.tolist())

norm per position: ['3046', '92', '89', '105', '85']
top outlier dims:  [447, 138, 378]


In [11]:
with hooks.ResidualHook(model, layer=LAYER) as hook:
    model.generate(**batch, max_new_tokens=8, do_sample=False,
                   pad_token_id=tokenizer.eos_token_id)

print(f"seq_lens        {hook.seq_lens}")
print(f"prefill         {hook.prefill_len} tokens")
print(f"decode steps    {hook.n_decode_steps}")

seq_lens        [5, 1, 1, 1, 1, 1, 1, 1]
prefill         5 tokens
decode steps    7


In [16]:
from steering import spaces

print("gpt2:      ", spaces.should_center("gpt2"))
print("gemma-2-it:", spaces.should_center("google/gemma-2-2b-it"))
try:
    spaces.should_center("some-model-nobody-measured")
except ValueError as e:
    print(f"\nrefused: {e}")

gpt2:       True
gemma-2-it: False

refused: unknown architecture 'some-model-nobody-measured': whether centering is output-neutral has not been measured for it. Measure the relative logit change under centering (see DEVLOG 2026-08-22) and add it to spaces._CENTERS before using this model.


Проверяем как работает hook

In [17]:
with hooks.ResidualHook(model, layer=LAYER, capture=True) as capture_hook, torch.no_grad():
    model(**batch)

h = capture_hook.captured[0]
scale = spaces.activation_scale(h, exclude_sink=True)
print(f"E||h|| (median, sink excluded): {scale:.1f}")

z = spaces.encode(h, scale=scale, center=True)
back = spaces.decode(z, scale=scale, center=True)
print("decode(encode(h)) == h:        ", torch.allclose(back, h))
print("decode(encode(h)) == center(h):", torch.allclose(back, spaces.center(h)))

E||h|| (median, sink excluded): 88.5
decode(encode(h)) == h:         False
decode(encode(h)) == center(h): True


## Загрузка SAE

In [18]:
from steering import vectors

sae = vectors.load_sae("gpt2-small-resid-post-v5-128k", "blocks.6.hook_resid_post",
                        layer=LAYER, d_model=768, device="cpu")
print(f"d_sae={sae.cfg.d_sae}  k={sae.cfg.k}  hook_name={sae.cfg.metadata.hook_name}")

try:
    vectors.load_sae("gpt2-small-resid-post-v5-128k", "blocks.6.hook_resid_post",
                      layer=7, d_model=768, device="cpu")
except ValueError as e:
    print(f"\nrefused: {e}")

d_sae=131072  k=32  hook_name=blocks.6.hook_resid_post

refused: SAE hook_name 'blocks.6.hook_resid_post' does not match intervention layer 7. Expected one of ['blocks.7.hook_resid_post', 'blocks.8.hook_resid_pre']. Remember resid_post(L) == resid_pre(L+1).


In [19]:
import glob
import pandas as pd

pq = glob.glob(str(io.REPO_ROOT / "../steering-final/**/pile-10k*/**/*.parquet"), recursive=True)
# or point this at wherever your Pile-10k parquet lives; any few hundred short texts will do
texts = pd.read_parquet(pq[0])["text"].head(200).tolist() if pq else None

if texts:
    center = spaces.should_center("gpt2")
    stats = vectors.compute_feature_stats(model, tokenizer, sae, layer=LAYER, texts=texts,
                                          batch_size=8, max_length=64, center=center)
    print(f"mean_l0 = {float(stats['mean_l0']):.2f}  (k = {sae.cfg.k})")
    lo, hi = vectors.band_from_mean(stats, lo=0.41, hi=20.5)
    print(f"band: [{lo:.2e}, {hi:.2e}]  -> {vectors.frequency_band(stats, lo, hi).numel()} in-band")

In [20]:
import torch as _torch

fake_stats = {"frequency": _torch.zeros(2000), "n_tokens": _torch.tensor(50_000)}
fake_stats["frequency"][200:800] = 2.44e-4  # 600 in-band candidates

split = vectors.select_and_freeze_split(
    fake_stats, io.RESULTS / "_smoke_split.json",
    freq_min=1e-4, freq_max=5e-3, n_dev=20, n_test=50, seed=0,
    model="gpt2", source="_smoke",
)
print(f"dev={len(split.dev)} test={len(split.test)} train_pool={len(split.train_pool())}")
print(f"fingerprint {split.fingerprint()}")

import json
p = io.RESULTS / "_smoke_split.json"
payload = json.loads(p.read_text())
payload["test"][0] = payload["test"][0] + 1  # tamper
p.write_text(json.dumps(payload))
try:
    vectors.load_split(p)
except ValueError as e:
    print(f"\nrefused: {e}")

for f in io.RESULTS.glob("_smoke_split.json"):
    f.unlink()

dev=20 test=50 train_pool=1930
fingerprint 7cc2a93c9fcda421

refused: HOLDOUT VIOLATION: split at /Users/bg/Coding/steering-mvp/results/_smoke_split.json does not match its fingerprint (be2cf35c2c422e1d != 7cc2a93c9fcda421). The frozen file was edited.


## Шаг 1. Проверка обычного steering'а

In [24]:
import json

FEATURE_IDS = [1878, 12789, 13836, 15452, 31544, 32801, 37444, 46144,
               53354, 60559, 60695, 65056, 67922, 74485, 89756, 130710]
prompts = json.loads((io.REPO_ROOT / "configs" / "prompts_neutral_32.json").read_text())["prompts"]
directions = vectors.steering_directions(sae, FEATURE_IDS)  # unit-norm, [16, 768]
print(len(prompts), "prompts,", directions.shape[0], "features")

32 prompts, 16 features


In [34]:
from steering import generate, interventions, metrics
import statistics

R_GRID = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
scale = spaces.activation_scale(capture_hook.captured[0], exclude_sink=True)

b0_out = generate.generate(model, tokenizer, prompts, interventions.NoSteering(),
                           layer=LAYER, device="cpu", max_new_tokens=32, batch_size=8, seed=0)
b0_nll = metrics.reference_nll(model, tokenizer, prompts, b0_out.texts, device="cpu")
b0_ppl = float(b0_nll[~b0_nll.isnan()].mean().exp())
print(f"B0 ppl = {b0_ppl:.2f}")

rows = []
for r in R_GRID:
    for fid, v in zip(FEATURE_IDS, directions):
        iv = interventions.NoSteering() if r == 0.0 else interventions.AdditiveSteering(v, r, scale)
        out = generate.generate(model, tokenizer, prompts, iv, layer=LAYER, device="cpu",
                                max_new_tokens=32, batch_size=8, seed=0)
        nll = metrics.reference_nll(model, tokenizer, prompts, out.texts, device="cpu")
        valid = ~nll.isnan()
        rows.append({
            "method": "B0" if r == 0.0 else "B1", "r": r, "feature_id": fid,
            "ppl": float(nll[valid].mean().exp()),
            "dist_2": metrics.distinct_n(out.texts, 2),
            "repetition_4": metrics.repetition_rate(out.texts, 4),
            "sample": out.texts[0],  # first prompt's continuation, for the eyeball check below
        })
    print(f"r={r} done")

import pandas as pd
df = pd.DataFrame(rows)
df["ppl_ratio"] = df["ppl"] / b0_ppl

print("\n--- aggregate (mean AND median")
print(df.groupby(["method", "r"])["ppl_ratio"].agg(["mean", "median", "max"]).round(2))

B0 ppl = 57.28
r=0.0 done
r=0.2 done
r=0.4 done
r=0.6 done
r=0.8 done
r=1 done

--- aggregate (mean AND median
             mean  median    max
method r                        
B0     0.0   1.00    1.00   1.00
B1     0.2   1.13    1.08   1.75
       0.4   1.85    1.71   2.89
       0.6   4.21    3.96   9.11
       0.8   8.07    5.94  27.10
       1.0  11.85    8.87  41.10


In [35]:
for r in [0.5, 1.0]:
    sub = df[df.r == r].sort_values("ppl_ratio", ascending=False)
    print(f"\n--- r={r} ---")
    print(sub[["feature_id", "ppl_ratio", "dist_2"]].to_string(index=False))


--- r=0.5 ---
Empty DataFrame
Columns: [feature_id, ppl_ratio, dist_2]
Index: []

--- r=1.0 ---
 feature_id  ppl_ratio   dist_2
     130710  41.096849 0.919643
      74485  20.150531 0.973958
      60559  19.588735 0.958438
      89756  18.170439 0.969559
      60695  11.658368 0.968468
       1878  11.223264 0.883005
      15452  11.017970 0.881963
      32801   9.099379 0.877551
      12789   8.632246 0.930521
      65056   8.487016 0.886946
      13836   6.461947 0.989201
      67922   5.419892 0.855911
      46144   5.174867 0.868932
      53354   5.139365 0.809080
      31544   4.860974 0.912556
      37444   3.359338 0.892768


In [36]:
import json
descr = json.loads(open("../steering-final/results/vector_extraction.json").read()) \
    if False else None

for fid in [67922, 60695]:
    v = directions[FEATURE_IDS.index(fid)]
    print(f"\n=== feature {fid} ===")
    for r in [0.0, 0.2, 0.4, 0.6, 0.8, 1]:
        iv = interventions.NoSteering() if r == 0.0 else interventions.AdditiveSteering(v, r, scale)
        out = generate.generate(model, tokenizer, prompts[:1], iv, layer=LAYER, device="cpu",
                                max_new_tokens=40, seed=0)
        print(f"  r={r:>4}: {out.texts[0]!r}")


=== feature 67922 ===
  r= 0.0: ' a button pilot is that you get to "tree" your UI on top of it and see what your user does in that specific can so far. You can plug in a scroll wheel, multi-'
  r= 0.2: ' a button pilot is that you can say "tree" and the birds that fly it would fly out of the sky in any case can sing a lot. Logdy also uses the heists algorithm to'
  r= 0.4: ' a button pilot is that you can say "tree" and the fully decentralized wiki is available. However, you have to be fairly canny. This Wiki is quite pretty big and he/she/'
  r= 0.6: ' a button-down encyclopedia is that is "tree" (legendary artifact for the game). wikipedia does not have an canon. Wikipedia Wiki is not in a position with the Sanskrit Sah'
  r= 0.8: ' a summary:\n\nOn Wik is "tree of bipartisan agreement on radical artifact, Wikipedia", while wikipedia is "Wikipedia is canon". Wikipedia Wiki is Wikipedia in a Standard Template Over Sanskrit Sah'
  r=   1: ' a summary:\n\nOn Wik is "tree on Wikipedia 

In [38]:
R_GRID = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

io.run_or_load("step1_gate_b0_b1",
                {"model": "gpt2", "layer": LAYER, "r_grid": R_GRID,
                 "feature_ids": FEATURE_IDS, "n_prompts": len(prompts),
                 "max_new_tokens": 32, "seed": 0, "version": 2},
                lambda: df, force=True)

computed step1_gate_b0_b1.csv  (385a4eb26165c8b8)


,method,r,feature_id,ppl,dist_2,repetition_4,sample,ppl_ratio
0,B0,0.0,1878,57.281975,0.974194,0.002604,"a ""surge"" that creates the illusion that any ...",1.000000
1,B0,0.0,12789,57.281975,0.974194,0.002604,"a ""surge"" that creates the illusion that any ...",1.000000
2,B0,0.0,13836,57.281975,0.974194,0.002604,"a ""surge"" that creates the illusion that any ...",1.000000
3,B0,0.0,15452,57.281975,0.974194,0.002604,"a ""surge"" that creates the illusion that any ...",1.000000
4,B0,0.0,31544,57.281975,0.974194,0.002604,"a ""surge"" that creates the illusion that any ...",1.000000
...,...,...,...,...,...,...,...,...
91,B1,1.0,65056,486.153015,0.886946,0.000000,"this exhibition, although it was recently ord...",8.487016
92,B1,1.0,67922,310.462128,0.855911,0.000000,"a ""PolyWorld"" was that the Wikipedia article ...",5.419892
93,B1,1.0,74485,1154.262207,0.973958,0.000000,"a ""special session of that"" ( of the any?) in...",20.150531
94,B1,1.0,89756,1040.838623,0.969559,0.000000,"a ""Phone"" Watch that Touch (or TV) Devices Be...",18.170439
